# Test de `build_solid_prop_config`

Módulo: `src.physics.thermodynamics.solid_props`  
Base de datos: `materials/solids/soliddb.txt`

## Propiedades devueltas

| Campo | Unidades | Descripción |
|---|---|---|
| `rho` | kg/m³ | Densidad |
| `cp` | J/kg·K | Calor específico másico |
| `k` | W/m·K | Conductividad térmica |
| `Tref` | K | Temperatura de referencia del polinomio |
| `Tmax` | K | Temperatura máxima de validez |
| `material` | str | Identificador del material |
| `mode` | str | Modo de evaluación (`constant`, `polynomial`, `fixed`) |

## Tres modos de operación

| Modo | Retorna | Uso típico |
|---|---|---|
| `constant` | Escalares en Tref | Propiedades fijas durante simulación |
| `polynomial` | Callables `f(T)` | Propiedades dependientes de T |
| `fixed` | Escalares del usuario | Prototipado / material no catalogado |

## Materiales disponibles en la BD

| id | Categoría | Tref | Tmax | ρ₀ [kg/m³] | cp₀ [J/kg·K] | k₀ [W/m·K] |
|---|---|---|---|---|---|---|
| SS316L | metal | 298 | 1273 | 7950 | 500 | 14.4 |
| P265GH | metal | 298 | 873 | 7850 | 490 | 51.0 |
| Inconel625 | metal | 298 | 1273 | 8440 | 410 | 9.8 |
| Al2O3 | ceramic | 298 | 1800 | 3960 | 777 | 30.0 |
| SiC | ceramic | 298 | 1773 | 3210 | 750 | 120.0 |
| SiO2_fused | ceramic | 298 | 1473 | 2200 | 740 | 1.38 |
| Cu | ceramic | 298 | 1273 | 8960 | 385 | 385.0 |

---

## Tests realizados

1. **Modo `constant` en Tref** — los 7 materiales; verifica que `rho`, `cp`, `k` coinciden con `a0`.
2. **Modo `constant` a T = 800 K** — verifica la evaluación fuera de Tref.
3. **Modo `polynomial`** — SS316L, Al2O3 y SiC en rejilla de temperatura.
4. **Modo `fixed`** — valores directos del usuario, sin acceso a la BD.
5. **Clipping de temperatura** — verifica que T < Tref devuelve el valor en Tref y T > Tmax devuelve el cap.

In [1]:
import os, sys
import numpy as np
import pandas as pd

ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from src.physics.thermodynamics.solid_props import build_solid_prop_config, eval_solid_property
from src.io.soliddb_reader import read_soliddb

DB_PATH = os.path.join(ROOT, "materials", "solids", "soliddb.txt")

MATERIALS_ALL = ["SS316L", "P265GH", "Inconel625", "Al2O3", "SiC", "SiO2_fused", "Cu"]

print("Import OK")
print("DB_PATH:", DB_PATH)
print("Existe DB:", os.path.isfile(DB_PATH))

Import OK
DB_PATH: C:\Users\MiguelCamaraSanz\OneDrive - Fundacion CIRCE\GITHUB\GasifierSimNet\materials\solids\soliddb.txt
Existe DB: True


## TEST 1 — Modo `constant` en Tref (los 7 materiales)

En modo `constant` las propiedades se devuelven como escalares evaluados en Tref,
lo que equivale al coeficiente `a0` de cada polinomio.
La columna `Tref check` verifica que `rho`, `cp` y `k` coinciden con los valores de referencia de la BD.

In [2]:
records = []
db = read_soliddb(DB_PATH)

for mat_id in MATERIALS_ALL:
    cfg = build_solid_prop_config(mat_id, mode="constant", db_path=DB_PATH)
    rec = db[mat_id]
    records.append({
        "material":    mat_id,
        "category":    rec.get("category", "-"),
        "Tref [K]":    cfg["Tref"],
        "Tmax [K]":    cfg["Tmax"],
        "rho [kg/m³]": cfg["rho"],
        "cp [J/kg·K]": cfg["cp"],
        "k [W/m·K]":   cfg["k"],
        "mode":        cfg["mode"],
    })

df1 = pd.DataFrame(records).set_index("material")
pd.set_option("display.float_format", "{:.5g}".format)
df1

,category,Tref [K],Tmax [K],rho [kg/m³],cp [J/kg·K],k [W/m·K],mode
material,,,,,,,
SS316L,metal,298,1273,7950,500,14.4,constant
P265GH,metal,298,873,7850,490,51,constant
Inconel625,metal,298,1273,8440,410,9.8,constant
Al2O3,ceramic,298,1800,3960,777,30,constant
SiC,ceramic,298,1773,3210,750,120,constant
SiO2_fused,ceramic,298,1473,2200,740,1.38,constant
Cu,metal,298,1273,8960,385,385,constant


## TEST 2 — Modo `constant` a T = 800 K

En modo `constant`, el parámetro `Temp` fija la temperatura de evaluación.
Todos los materiales del catálogo tienen `Tmax ≥ 800 K`, por lo que la evaluación
está dentro del rango de validez de todos los polinomios.

In [3]:
T_EVAL = 800.0   # [K]

records2 = []
for mat_id in MATERIALS_ALL:
    # Modo polynomial para evaluar a T arbitraria
    cfg = build_solid_prop_config(mat_id, mode="polynomial", db_path=DB_PATH)
    T_use = min(T_EVAL, cfg["Tmax"])   # respetar Tmax de cada material
    records2.append({
        "material":       mat_id,
        "T eval [K]":     T_use,
        "Tmax [K]":       cfg["Tmax"],
        "rho [kg/m³]":    eval_solid_property(cfg["rho"], T_use),
        "cp [J/kg·K]":    eval_solid_property(cfg["cp"],  T_use),
        "k [W/m·K]":      eval_solid_property(cfg["k"],   T_use),
        "Δrho/rho₀ [%]": (eval_solid_property(cfg["rho"], T_use) - eval_solid_property(cfg["rho"], cfg["Tref"])) / eval_solid_property(cfg["rho"], cfg["Tref"]) * 100,
        "Δcp/cp₀  [%]": (eval_solid_property(cfg["cp"],  T_use) - eval_solid_property(cfg["cp"],  cfg["Tref"])) / eval_solid_property(cfg["cp"],  cfg["Tref"]) * 100,
        "Δk/k₀    [%]": (eval_solid_property(cfg["k"],   T_use) - eval_solid_property(cfg["k"],   cfg["Tref"])) / eval_solid_property(cfg["k"],   cfg["Tref"]) * 100,
    })

df2 = pd.DataFrame(records2).set_index("material")
df2

,T eval [K],Tmax [K],rho [kg/m³],cp [J/kg·K],k [W/m·K],Δrho/rho₀ [%],Δcp/cp₀ [%],Δk/k₀ [%]
material,,,,,,,,
SS316L,800,1273,7758.2,561.79,19.564,-2.4121,12.357,35.862
P265GH,800,873,7707.9,568.77,37.031,-1.81,16.076,-27.39
Inconel625,800,1273,8277.4,484.66,17.157,-1.9265,18.209,75.075
Al2O3,800,1800,3912.3,1141.7,17.46,-1.2048,46.931,-41.8
SiC,800,1773,3190.7,1011.9,60.433,-0.6024,34.926,-49.639
SiO2_fused,800,1473,2198.2,847.2,1.626,-0.08283,14.486,17.825
Cu,800,1273,8730.6,397.87,363.89,-2.5602,3.3433,-5.483


## TEST 3 — Modo `polynomial` (SS316L, Al2O3, SiC en rejilla de T)

En modo `polynomial` las propiedades se devuelven como callables `f(T)` que aceptan
tanto escalares como arrays de numpy. Se muestran los tres materiales en su rango
completo `[Tref, Tmax]` con 8 puntos equiespaciados.

In [4]:
MATERIALS_POLY = ["SS316L", "Al2O3", "SiC"]
N_POINTS = 8

records3 = []
for mat_id in MATERIALS_POLY:
    cfg = build_solid_prop_config(mat_id, mode="polynomial", db_path=DB_PATH)
    T_grid = np.linspace(cfg["Tref"], cfg["Tmax"], N_POINTS)

    for T in T_grid:
        records3.append({
            "material":    mat_id,
            "T [K]":       round(float(T), 1),
            "rho [kg/m³]": eval_solid_property(cfg["rho"], T),
            "cp [J/kg·K]": eval_solid_property(cfg["cp"],  T),
            "k [W/m·K]":   eval_solid_property(cfg["k"],   T),
        })

df3 = pd.DataFrame(records3).set_index(["material", "T [K]"])
df3

rho [kg/m³]  cp [J/kg·K]  k [W/m·K]
material T [K]                                      
SS316L   298            7950          500       14.4
         437.3        7896.8       517.14     15.833
         576.6        7843.6       534.29     17.266
         715.9        7790.4       551.43     18.698
         855.1        7737.2       568.57     20.131
         994.4          7684       585.72     21.564
         1133.7       7630.8       602.86     22.997
         1273         7577.6          620      24.43
Al2O3    298            3960          777         30
         512.6        3939.6       948.25     24.023
         727.1        3919.2       1096.5     18.967
         941.7        3898.8       1221.8     14.832
         1156.3       3878.4       1324.2     11.618
         1370.9         3858       1403.6     9.3245
         1585.4       3837.6         1460     7.9519
         1800         3817.2       1493.5        7.5
SiC      298            3210          750        120
         508.7        3201.9       868.96     92.323
         719.4        3193.8       974.89     68.478
         930.1        3185.6       1067.8     48.747
         1140.9       3177.5       1147.6      33.41
         1351.6       3169.4       1214.5     22.748
         1562.3       3161.3       1268.2     17.041
         1773         3153.2         1309      16.57

### Verificación: `eval_solid_property` acepta arrays

Comprueba que pasar un array de temperaturas devuelve un array de igual shape,
sin bucle explícito en el código llamante.

In [5]:
cfg_ss = build_solid_prop_config("SS316L", mode="polynomial", db_path=DB_PATH)
T_arr  = np.array([300., 500., 700., 900., 1100., 1273.])   # K

rho_arr = eval_solid_property(cfg_ss["rho"], T_arr)
cp_arr  = eval_solid_property(cfg_ss["cp"],  T_arr)
k_arr   = eval_solid_property(cfg_ss["k"],   T_arr)

df3b = pd.DataFrame({
    "T [K]":        T_arr,
    "rho [kg/m³]": rho_arr,
    "cp [J/kg·K]": cp_arr,
    "k [W/m·K]":   k_arr,
})
print(f"Input shape: {T_arr.shape}  →  output shapes: rho={rho_arr.shape}, cp={cp_arr.shape}, k={k_arr.shape}")
df3b

Input shape: (6,)  →  output shapes: rho=(6,), cp=(6,), k=(6,)


,T [K],rho [kg/m³],cp [J/kg·K],k [W/m·K]
0,300,7949.2,500.25,14.421
1,500,7872.8,524.86,16.478
2,700,7796.4,549.48,18.535
3,900,7720,574.09,20.593
4,1100,7643.6,598.71,22.65
5,1273,7577.6,620,24.43


## TEST 4 — Modo `fixed` (valores directos del usuario)

En modo `fixed` no se accede a la BD. El usuario suministra directamente los escalares
`rho_fixed`, `cp_fixed` y `k_fixed`. Útil para parámetros de ajuste o materiales
no catalogados.

In [6]:
cfg_fixed = build_solid_prop_config(
    material_id = "SS316L",   # ignorado en modo fixed
    mode        = "fixed",
    rho_fixed   = 8000.0,     # kg/m³  — valor de ajuste hipotético
    cp_fixed    = 520.0,      # J/kg·K
    k_fixed     = 16.0,       # W/m·K
)

print("Modo fixed:")
print(f"  material : {cfg_fixed['material']}")
print(f"  mode     : {cfg_fixed['mode']}")
print(f"  Tref     : {cfg_fixed['Tref']}   (None — sin restricción de rango)")
print(f"  Tmax     : {cfg_fixed['Tmax']}")
print(f"  rho      : {cfg_fixed['rho']} kg/m³")
print(f"  cp       : {cfg_fixed['cp']} J/kg·K")
print(f"  k        : {cfg_fixed['k']} W/m·K")
print()

# eval_solid_property con modo fixed devuelve el escalar (o array constante)
T_test = np.array([300., 700., 1200.])
print("eval_solid_property con fixed a T =", T_test, "K")
print("  rho:", eval_solid_property(cfg_fixed["rho"], T_test))
print("  cp: ", eval_solid_property(cfg_fixed["cp"],  T_test))
print("  k:  ", eval_solid_property(cfg_fixed["k"],   T_test))

Modo fixed:
  material : fixed
  mode     : fixed
  Tref     : None   (None — sin restricción de rango)
  Tmax     : None
  rho      : 8000.0 kg/m³
  cp       : 520.0 J/kg·K
  k        : 16.0 W/m·K

eval_solid_property con fixed a T = [ 300.  700. 1200.] K
  rho: [8000. 8000. 8000.]
  cp:  [520. 520. 520.]
  k:   [16. 16. 16.]


## TEST 5 — Clipping de temperatura

El polinomio se evalúa con T recortada a `[Tref, Tmax]`. Verifica que:
- `T < Tref` → devuelve el valor en Tref (igual que `a0`).
- `T > Tmax` → devuelve el valor en Tmax (igual que `cap_at_tmax`).
- Dentro del rango → interpolación polinómica normal.

In [7]:
cfg_ss = build_solid_prop_config("SS316L", mode="polynomial", db_path=DB_PATH)
Tref_ss = cfg_ss["Tref"]   # 298 K
Tmax_ss = cfg_ss["Tmax"]   # 1273 K

T_below = 100.0    # K — por debajo de Tref
T_above = 1600.0   # K — por encima de Tmax
T_in    = 700.0    # K — dentro del rango

cap = read_soliddb(DB_PATH)["SS316L"]["cap_at_tmax"]

rows = []
for T_val, label in [(T_below, f"T={T_below:.0f} K (< Tref)"),
                     (Tref_ss,  f"T=Tref={Tref_ss:.0f} K"),
                     (T_in,     f"T={T_in:.0f} K (en rango)"),
                     (Tmax_ss,  f"T=Tmax={Tmax_ss:.0f} K"),
                     (T_above,  f"T={T_above:.0f} K (> Tmax)")]:
    rows.append({
        "T evaluada":    label,
        "rho [kg/m³]": eval_solid_property(cfg_ss["rho"], T_val),
        "cp [J/kg·K]": eval_solid_property(cfg_ss["cp"],  T_val),
        "k [W/m·K]":   eval_solid_property(cfg_ss["k"],   T_val),
    })

df5 = pd.DataFrame(rows).set_index("T evaluada")

print(f"SS316L  Tref={Tref_ss:.0f} K  Tmax={Tmax_ss:.0f} K")
print(f"cap_at_tmax → rho={cap['rho_kg_per_m3']:.2f}  cp={cap['cp_J_per_kgK']:.2f}  k={cap['k_W_per_mK']:.4f}")
print()
df5

SS316L  Tref=298 K  Tmax=1273 K
cap_at_tmax → rho=7577.50  cp=619.90  k=24.4300



,rho [kg/m³],cp [J/kg·K],k [W/m·K]
T evaluada,,,
T=100 K (< Tref),7950,500,14.4
T=Tref=298 K,7950,500,14.4
T=700 K (en rango),7796.4,549.48,18.535
T=Tmax=1273 K,7577.6,620,24.43
T=1600 K (> Tmax),7577.6,620,24.43


## TEST 6 — Comparación de los tres modos para SS316L

Muestra en una única tabla los valores de `rho`, `cp` y `k` obtenidos con cada modo
para verificar la coherencia entre ellos.

In [8]:
T_CMP = 600.0   # K — temperatura de comparación

cfg_const = build_solid_prop_config("SS316L", mode="constant",   db_path=DB_PATH)
cfg_poly  = build_solid_prop_config("SS316L", mode="polynomial",  db_path=DB_PATH)
cfg_fix   = build_solid_prop_config(
    "SS316L", mode="fixed",
    rho_fixed=7834.6, cp_fixed=537.2, k_fixed=17.51,   # valores aprox. de poly @ 600 K
)

df6 = pd.DataFrame([
    {
        "modo":        "constant  (en Tref=298 K)",
        "rho [kg/m³]": cfg_const["rho"],
        "cp [J/kg·K]": cfg_const["cp"],
        "k [W/m·K]":   cfg_const["k"],
        "nota":        "escalar fijo independiente de T",
    },
    {
        "modo":        f"polynomial (evaluado a {T_CMP:.0f} K)",
        "rho [kg/m³]": eval_solid_property(cfg_poly["rho"], T_CMP),
        "cp [J/kg·K]": eval_solid_property(cfg_poly["cp"],  T_CMP),
        "k [W/m·K]":   eval_solid_property(cfg_poly["k"],   T_CMP),
        "nota":        "callable f(T), T recortada a [298, 1273] K",
    },
    {
        "modo":        "fixed     (valores directos)",
        "rho [kg/m³]": cfg_fix["rho"],
        "cp [J/kg·K]": cfg_fix["cp"],
        "k [W/m·K]":   cfg_fix["k"],
        "nota":        "escalar del usuario, sin acceso a BD",
    },
]).set_index("modo")

df6

,rho [kg/m³],cp [J/kg·K],k [W/m·K],nota
modo,,,,
constant (en Tref=298 K),7950,500,14.4,escalar fijo independiente de T
polynomial (evaluado a 600 K),7834.6,537.17,17.507,"callable f(T), T recortada a [298, 1273] K"
fixed (valores directos),7834.6,537.2,17.51,"escalar del usuario, sin acceso a BD"


## TEST 7 — Verificación de error ante material inexistente

Comprueba que `build_solid_prop_config` lanza `KeyError` si se solicita un material
que no está en la BD, con un mensaje que lista los disponibles.

In [9]:
try:
    build_solid_prop_config("TitanioGrado5", mode="constant", db_path=DB_PATH)
    print("ERROR: debería haber lanzado KeyError")
except KeyError as e:
    print("KeyError capturado correctamente:")
    print(" ", e)

print()

# Modo fixed con parámetros faltantes → ValueError
try:
    build_solid_prop_config("cualquiera", mode="fixed", rho_fixed=7900.0)
    print("ERROR: debería haber lanzado ValueError")
except ValueError as e:
    print("ValueError capturado correctamente:")
    print(" ", e)

KeyError capturado correctamente:
  "Material 'TitanioGrado5' not found in soliddb at 'C:\\Users\\MiguelCamaraSanz\\OneDrive - Fundacion CIRCE\\GITHUB\\GasifierSimNet\\materials\\solids\\soliddb.txt'. Available: ['SS316L', 'P265GH', 'Inconel625', 'Al2O3', 'SiC', 'SiO2_fused', 'Cu']"

ValueError capturado correctamente:
  mode='fixed' requires rho_fixed, cp_fixed and k_fixed to be provided.
